# NB-2b — FT-TRAJ-1: two additional LRAA full-fine-tuning trajectories with per-epoch three-state evaluation

Implements manifest amendment v1.3 (A.12). Settings identical to the validated control and to NB-2. Seeds **7595, 1676**
(drawn from `default_rng(20260925)`, recorded before training). After every epoch the held-out set is scored in the three states
(`on`, `norm_only`, `off`) with predictions, probabilities, gate value and relative branch norm — descriptive only; checkpoint
selection stays on internal validation. The selected epoch's full state is saved to Drive with SHA-256. No STOP comparison with the
archive (no archived counterpart). The analysis combines these trajectories with the NB-2 rerun (read from `MyDrive/FMI/NB2/progress/`)
and the archived control to list the token-mixing component A_FT of every full-fine-tuning trajectory.

Restart-safe per run; estimate about 1.7 A100 hours. The runtime is not deleted.

In [ ]:
# 1) Drive first; start time.
import time
from datetime import datetime, timezone
RUN_STARTED_PERF = time.perf_counter(); RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
from google.colab import drive
drive.mount('/content/drive')
print('Run started (UTC):', RUN_STARTED_UTC)

In [ ]:
# 2) Pinned dependencies (as in the archived validated-control notebook).
import subprocess, sys
packages = ['pandas==2.2.3', 'transformers==4.57.6', 'datasets==3.2.0', 'huggingface-hub==0.36.2', 'accelerate>=1.10,<2',
            'sentencepiece>=0.2,<0.3', 'protobuf>=5.29,<6', 'scikit-learn==1.6.1']
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *packages], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if proc.returncode != 0:
    print(proc.stdout); raise RuntimeError('Dependency installation failed.')
print('Dependencies installed.')

In [ ]:
# 3) Imports, GPU, locked configuration, paths, base model copy, repository clone (archived values for gates).
import gc, hashlib, json, math, os, random, shutil, warnings, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import DatasetDict, load_dataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
warnings.filterwarnings('ignore', category=FutureWarning)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'; os.environ['WANDB_DISABLED'] = 'true'
try:
    from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not torch.cuda.is_available(): raise RuntimeError('GPU required. An A100 is recommended; a T4 works but is roughly 3–4× slower.')
DEVICE = torch.device('cuda')
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True; torch.set_float32_matmul_precision('high')
GPU_NAME = torch.cuda.get_device_name(0); print('GPU:', GPU_NAME, '| autocast', AMP_DTYPE, '| torch', torch.__version__, '| CUDA', torch.version.cuda)

MANIFEST_ID = 'FMI manifest v1.3 amendment / A.12 FT-TRAJ-1 (per-epoch three-state evaluation)'
MANIFEST_LOCK_COMMIT = ''   # from MANIFEST_LOCK.md; recorded, not checked
BOOTSTRAP_SEED, BOOTSTRAP_REPLICATES = 20260924, 10000
STOP_GATE_POINTS = 1.0

# ---- LOCKED configuration (identical to the archived validated control) ----
SEEDS = [7595, 1676]   # FT-TRAJ-1 seeds drawn from default_rng(20260925), recorded in the v1.3 amendment
DATA_SEED = 2026; EPOCHS = 3; MAX_LENGTH = 512; LATENT_RANK = 256; LATENT_HEADS = 8
BATCH_SIZE = 2; EVAL_BATCH_SIZE = 4; GRAD_ACCUM_STEPS = 8; ENCODER_LR = 8e-6; ADAPTER_SCORER_LR = 2e-4
WEIGHT_DECAY = 0.01; WARMUP_RATIO = 0.06; VAL_TARGET = 1075
ARMS = ['lraa']   # FT-TRAJ-1: LRAA arm only

DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/Colab Notebooks/NLP Modeller/deberta-v3-large')
LOCAL_MODEL_DIR = Path('/content/models/deberta-v3-large')
WORK_DIR = Path('/content/fmi_nb2'); RESULT_DIR = WORK_DIR / 'results'; REPO_DIR = WORK_DIR / 'repo'
DRIVE_ROOT = Path('/content/drive/MyDrive/FMI/NB2b_FT-TRAJ-1'); PROGRESS_DIR = DRIVE_ROOT / 'progress'; FULL_STATE_DIR = DRIVE_ROOT / 'full_states'
NB1_DIR = Path('/content/drive/MyDrive/FMI/D-NORM-1')
for p in (WORK_DIR, RESULT_DIR, PROGRESS_DIR, FULL_STATE_DIR): p.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
def sha256_file(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()

required = ['config.json', 'spm.model', 'tokenizer_config.json']
missing = [n for n in required if not (DRIVE_MODEL_DIR / n).is_file()]
has_weights = any((DRIVE_MODEL_DIR / n).is_file() for n in ('model.safetensors', 'pytorch_model.bin'))
if missing or not has_weights: raise FileNotFoundError(f'Incomplete base model folder: {DRIVE_MODEL_DIR}; missing={missing}; weights={has_weights}')
shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, dirs_exist_ok=True); MODEL_NAME = str(LOCAL_MODEL_DIR)

# archived validated-control values for the STOP gate (exact, from the public repository)
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/kurbankotan/Adapter-Contribution-Audit.git', str(REPO_DIR)], text=True, capture_output=True)
if r.returncode != 0: print(r.stderr); raise RuntimeError('git clone failed')
REPO_COMMIT = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True, capture_output=True).stdout.strip()
def find_one(pattern, under=REPO_DIR):
    hits = sorted(Path(under).rglob(pattern), key=lambda p: len(p.parts))
    if not hits: raise FileNotFoundError(pattern)
    return hits[0]
ARCH_V12 = pd.read_csv(find_one('adapter_on_off_run_results.csv', REPO_DIR / 'results' / 'v12_validated_full_finetuning'))
ARCHIVED_FT_BA = {(int(s), st): 100 * float(v) for s, st, v in zip(ARCH_V12.seed, ARCH_V12.adapter_state, ARCH_V12.balanced_accuracy)}
FROZEN_HEAD_ONLY_MEAN_BA = 0.6046690798271527
print('Archived validated-control BA (on/off), seeds 13/42/71:', {s: (round(ARCHIVED_FT_BA[(s, 'on')], 2), round(ARCHIVED_FT_BA[(s, 'off')], 2)) for s in (13, 42, 71)})

# NB-1 frozen LRAA values (A_frozen per seed); prefer the NB-1 result ZIP on Drive, else the embedded archived numbers
NB1_A_FROZEN = {13: 28.652755, 42: 23.708308, 71: 30.375045}
NB1_ON_OFF_FROZEN = {13: 28.498131676170992, 42: 24.11058282080514, 71: 30.23795729397838}
NB1_PRED = None
nb1_zips = sorted(NB1_DIR.glob('fmi_dnorm1_frozen_results_*.zip'))
if nb1_zips:
    with zipfile.ZipFile(nb1_zips[-1]) as zf:
        d = pd.read_csv(zf.open('decomposition_by_seed.csv')); d = d[d.model == 'lcca']
        NB1_A_FROZEN = {int(s): float(a) for s, a in zip(d.seed, d.A_on_minus_norm_only)}
        NB1_PRED = pd.read_csv(zf.open('three_state_predictions.csv')); NB1_PRED = NB1_PRED[NB1_PRED.model == 'lcca']
    print('NB-1 values loaded from', nb1_zips[-1].name)
else:
    print('NB-1 ZIP not found on Drive; using embedded NB-1 values (seed-level ΔA only, no cluster bootstrap for ΔA).')
locked = {'experiment': 'FMI NB-2 FT-RERUN-1 + D2', 'manifest': MANIFEST_ID, 'manifest_lock_commit': MANIFEST_LOCK_COMMIT, 'repository_commit': REPO_COMMIT,
          'arms': ARMS, 'seeds': SEEDS, 'epochs': EPOCHS, 'max_length': MAX_LENGTH, 'latent_rank': LATENT_RANK, 'latent_heads': LATENT_HEADS,
          'batch_size': BATCH_SIZE, 'grad_accum': GRAD_ACCUM_STEPS, 'encoder_lr': ENCODER_LR, 'adapter_scorer_lr': ADAPTER_SCORER_LR,
          'weight_decay': WEIGHT_DECAY, 'warmup_ratio': WARMUP_RATIO, 'stop_gate_points': STOP_GATE_POINTS, 'nb1_A_frozen': NB1_A_FROZEN}
(RESULT_DIR / 'locked_config_before_training.json').write_text(json.dumps(locked, indent=2)); print(json.dumps(locked, indent=2))

In [ ]:
# 4) Exact data construction (unchanged): passage-grouped split, official validation as held-out, tokenization.
seed_everything(DATA_SEED)
raw = load_dataset('google/boolq', token=HF_TOKEN); train_full = raw['train']; heldout_raw = raw['validation']
groups = np.asarray(train_full['passage']); labels = np.asarray(train_full['answer'], dtype=int)
splitter = GroupShuffleSplit(n_splits=80, test_size=VAL_TARGET / len(train_full), random_state=DATA_SEED)
global_rate = labels.mean(); best_split = None
for tr_idx, va_idx in splitter.split(np.zeros(len(labels)), labels, groups):
    score = abs(len(va_idx) - VAL_TARGET) / max(1, VAL_TARGET) + abs(labels[va_idx].mean() - global_rate)
    if best_split is None or score < best_split[0]: best_split = (score, tr_idx, va_idx)
_, train_idx, val_idx = best_split
train_raw = train_full.select(train_idx.tolist()); val_raw = train_full.select(val_idx.tolist())
assert set(train_raw['passage']).isdisjoint(set(val_raw['passage']))
assert len(train_raw) == 8351 and len(val_raw) == 1076 and len(heldout_raw) == 3270
GOLD = np.array([int(bool(a)) for a in heldout_raw['answer']])
_h = [hashlib.sha256(p.encode('utf-8')).hexdigest() for p in heldout_raw['passage']]; _cl = {h: i for i, h in enumerate(sorted(set(_h)))}
CLUSTER = np.array([_cl[h] for h in _h]); N_CLUSTERS = len(_cl); print('held-out rows', len(GOLD), '| passage clusters', N_CLUSTERS)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)
def encode_dataset(ds, desc):
    def enc(batch):
        qs = [f'question: {q} answer:' for q in batch['question']]
        yes = tokenizer([q + ' yes' for q in qs], batch['passage'], truncation='only_second', max_length=MAX_LENGTH)
        no = tokenizer([q + ' no' for q in qs], batch['passage'], truncation='only_second', max_length=MAX_LENGTH)
        return {'yes_input_ids': yes['input_ids'], 'yes_attention_mask': yes['attention_mask'], 'no_input_ids': no['input_ids'], 'no_attention_mask': no['attention_mask']}
    return ds.map(enc, batched=True, batch_size=256, remove_columns=['question', 'passage'], desc=desc)
encoded = DatasetDict({'train': encode_dataset(train_raw, 'train'), 'validation': encode_dataset(val_raw, 'validation'), 'evaluation': encode_dataset(heldout_raw, 'held-out')})

class PairDataset(torch.utils.data.Dataset):
    def __init__(self, ds, training=False, seed=0): self.ds, self.training, self.seed, self.epoch = ds, training, int(seed), 0
    def set_epoch(self, e): self.epoch = int(e)
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        row = self.ds[int(idx)]; swap = False
        if self.training: swap = random.Random(self.seed * 1000003 + self.epoch * 10007 + int(idx)).random() < 0.5
        semantics = [0, 1] if swap else [1, 0]; prefix = ['no', 'yes'] if swap else ['yes', 'no']
        cands = [{'input_ids': row[f'{p}_input_ids'], 'attention_mask': row[f'{p}_attention_mask']} for p in prefix]
        gold = int(row['answer'])
        return {'candidates': cands, 'candidate_semantics': semantics, 'labels': semantics.index(gold), 'semantic_labels': gold}
def pair_collate(batch):
    flat = [c for item in batch for c in item['candidates']]; padded = tokenizer.pad(flat, padding=True, return_tensors='pt'); bsz = len(batch)
    return {'input_ids': padded['input_ids'].view(bsz, 2, -1), 'attention_mask': padded['attention_mask'].view(bsz, 2, -1),
            'candidate_semantics': torch.tensor([x['candidate_semantics'] for x in batch]), 'labels': torch.tensor([x['labels'] for x in batch]),
            'semantic_labels': torch.tensor([x['semantic_labels'] for x in batch])}
def make_loader(ds, seed, training=False):
    pds = PairDataset(ds, training=training, seed=seed); g = torch.Generator().manual_seed(seed)
    return pds, DataLoader(pds, batch_size=BATCH_SIZE if training else EVAL_BATCH_SIZE, shuffle=training, collate_fn=pair_collate, num_workers=2, pin_memory=True, generator=g)
yes_count = int(np.sum(encoded['train']['answer'])); no_count = len(encoded['train']) - yes_count
class_weight = torch.tensor([len(encoded['train']) / (2 * max(no_count, 1)), len(encoded['train']) / (2 * max(yes_count, 1))], dtype=torch.float32, device=DEVICE)
print('Tokenization complete. Class weights (no, yes):', class_weight.tolist())

In [ ]:
# 5) Models: exact LRAA (three intervention states) on a trainable encoder, and the no-adapter arm.
class LatentRelationAdapter(nn.Module):
    def __init__(self, hidden_size, rank, heads, dropout=0.1):
        super().__init__(); assert rank % heads == 0
        self.rank, self.heads, self.head_dim = rank, heads, rank // heads
        self.q = nn.Linear(hidden_size, rank, bias=False); self.k = nn.Linear(hidden_size, rank, bias=False)
        self.v = nn.Linear(hidden_size, rank, bias=False); self.out = nn.Linear(rank, hidden_size, bias=False)
        self.norm = nn.LayerNorm(hidden_size); self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))
    @property
    def alpha(self): return torch.sigmoid(self.alpha_logit)
    def branch(self, hidden, attention_mask):
        bsz, seq_len, _ = hidden.shape
        split = lambda x: x.view(bsz, seq_len, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = split(self.q(hidden)), split(self.k(hidden)), split(self.v(hidden))
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(attention_mask[:, None, None, :] == 0, torch.finfo(scores.dtype).min)
        z = torch.matmul(torch.softmax(scores, dim=-1), v).transpose(1, 2).contiguous().view(bsz, seq_len, self.rank)
        return self.out(z)
    def forward(self, hidden, attention_mask, state='on'):
        if state == 'off': return hidden
        if state == 'norm_only': return self.norm(hidden)
        return self.norm(hidden + self.alpha * self.dropout(self.branch(hidden, attention_mask)))

class FullFineTuned(nn.Module):
    def __init__(self, arm):
        super().__init__(); self.arm = arm
        self.encoder = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN)
        if hasattr(self.encoder.config, 'use_cache'): self.encoder.config.use_cache = False
        self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        self.hidden_size = int(self.encoder.config.hidden_size)
        self.adapter = LatentRelationAdapter(self.hidden_size, LATENT_RANK, LATENT_HEADS) if arm == 'lraa' else None
        self.dropout = nn.Dropout(0.15); self.scorer = nn.Linear(self.hidden_size, 1)
    def score_hidden(self, hidden, attention_mask, state='on'):
        if self.adapter is not None: hidden = self.adapter(hidden, attention_mask, state=state)
        return self.scorer(self.dropout(hidden[:, 0]))
    def forward(self, input_ids, attention_mask, state='on'):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.score_hidden(hidden, attention_mask, state)
    @torch.no_grad()
    def branch_ratio_cls(self, hidden, attention_mask):
        z = self.adapter.branch(hidden, attention_mask)[:, 0].float() * self.adapter.alpha.float()
        return z.norm(dim=-1) / hidden[:, 0].float().norm(dim=-1).clamp_min(1e-6)

probe = FullFineTuned('lraa')
pc = {'encoder': sum(p.numel() for p in probe.encoder.parameters()), 'adapter': sum(p.numel() for p in probe.adapter.parameters()), 'scorer': sum(p.numel() for p in probe.scorer.parameters())}
assert pc['adapter'] == 1050625 and pc['scorer'] == 1025
pd.DataFrame([pc]).to_csv(RESULT_DIR / 'parameter_counts.csv', index=False); print(pc); del probe; gc.collect()

In [ ]:
# 6) Evaluation (multi-state, with probabilities and D4) and the locked training loop.
def sem_metrics(gold, pred, prob, loss):
    return {'loss': float(loss), 'accuracy': accuracy_score(gold, pred), 'balanced_accuracy': balanced_accuracy_score(gold, pred),
            'macro_f1': f1_score(gold, pred, average='macro', zero_division=0), 'predicted_yes_rate': float(np.mean(pred)), 'true_yes_rate': float(np.mean(gold)),
            'log_loss': float(log_loss(gold, np.clip(prob, 1e-7, 1 - 1e-7))) if prob is not None else None}

@torch.no_grad()
def evaluate_states(model, loader, states, collect_branch=False):
    model.eval(); tot = {st: {'loss_sum': 0.0, 'gold': [], 'pred': [], 'prob': []} for st in states}; ratios = []
    for batch in loader:
        ids = batch['input_ids'].to(DEVICE, non_blocking=True); mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE); sem = batch['candidate_semantics']; gold = batch['semantic_labels'].numpy(); bsz, nopt, L = ids.shape
        flat_ids, flat_mask = ids.view(bsz * nopt, L), mask.view(bsz * nopt, L)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            hidden = model.encoder(input_ids=flat_ids, attention_mask=flat_mask).last_hidden_state
            for st in states:
                logits = model.score_hidden(hidden, flat_mask, state=st).view(bsz, nopt).float()
                loss = F.cross_entropy(logits, labels); probs = torch.softmax(logits, dim=-1).cpu()
                pred_idx = logits.argmax(dim=-1).cpu(); sem_pred = sem.gather(1, pred_idx[:, None]).squeeze(1).numpy()
                p_yes = probs.gather(1, (sem == 1).long().argmax(dim=1, keepdim=True)).squeeze(1).numpy()
                tot[st]['loss_sum'] += loss.item() * bsz; tot[st]['gold'].extend(gold.tolist()); tot[st]['pred'].extend(sem_pred.tolist()); tot[st]['prob'].extend(p_yes.tolist())
            if collect_branch and model.adapter is not None: ratios.extend(model.branch_ratio_cls(hidden, flat_mask).cpu().numpy().tolist())
    out = {}
    for st, v in tot.items():
        g, p, pr = np.asarray(v['gold']), np.asarray(v['pred']), np.asarray(v['prob'])
        out[st] = {**sem_metrics(g, p, pr, v['loss_sum'] / len(g)), 'gold': g, 'pred': p, 'prob_yes': pr}
    return out, (float(np.mean(ratios)) if ratios else None)

def cpu_state_dict(m): return {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}

def train_run(arm, seed):
    seed_everything(seed); model = FullFineTuned(arm).to(DEVICE)
    train_ds, train_loader = make_loader(encoded['train'], seed, training=True); _, val_loader = make_loader(encoded['validation'], seed, training=False)
    _, heldout_loader_ep = make_loader(encoded['evaluation'], seed, training=False)
    groups_ = [{'params': list(model.encoder.parameters()), 'lr': ENCODER_LR, 'name': 'encoder'}]
    head_params = list(model.scorer.parameters()) + (list(model.adapter.parameters()) if model.adapter is not None else [])
    groups_.append({'params': head_params, 'lr': ADAPTER_SCORER_LR, 'name': 'adapter_and_scorer'})
    optimizer = torch.optim.AdamW(groups_, weight_decay=WEIGHT_DECAY)
    updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS); total_updates = updates_per_epoch * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_updates * WARMUP_RATIO), total_updates)
    scaler = torch.amp.GradScaler('cuda', enabled=(AMP_DTYPE == torch.float16)); optimizer.zero_grad(set_to_none=True)
    best = {'key': None, 'state': None, 'epoch': None, 'metrics': None}; history = []; start = time.time(); torch.cuda.reset_peak_memory_stats()
    for epoch in range(1, EPOCHS + 1):
        train_ds.set_epoch(epoch); model.train(); stats = {'loss': 0.0, 'correct': 0, 'seen': 0}
        for step, batch in enumerate(train_loader, 1):
            ids = batch['input_ids'].to(DEVICE, non_blocking=True); mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
            labels = batch['labels'].to(DEVICE); semantic = batch['semantic_labels'].to(DEVICE); bsz, nopt, L = ids.shape
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(ids.view(bsz * nopt, L), mask.view(bsz * nopt, L), state='on').view(bsz, nopt)
                per_item = F.cross_entropy(logits.float(), labels, reduction='none'); loss = (per_item * class_weight[semantic]).mean()
            if not torch.isfinite(loss): raise FloatingPointError(f'Non-finite loss at arm={arm}, seed={seed}, epoch={epoch}, step={step}')
            scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
            stats['loss'] += loss.item() * bsz; stats['correct'] += (logits.argmax(-1) == labels).sum().item(); stats['seen'] += bsz
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True); scheduler.step()
            if step % 500 == 0: print(f'{arm} seed {seed} epoch {epoch}/{EPOCHS} batch {step}/{len(train_loader)}')
        val, _ = evaluate_states(model, val_loader, ['on']); val = val['on']
        row = {'arm': arm, 'seed': seed, 'epoch': epoch, 'train_loss': stats['loss'] / stats['seen'], 'train_accuracy': stats['correct'] / stats['seen'],
               **{f'val_{k}': v for k, v in val.items() if k not in ('gold', 'pred', 'prob_yes')}}
        history.append(row); print({k: (round(v, 5) if isinstance(v, float) else v) for k, v in row.items()})
        # FT-TRAJ-1: descriptive per-epoch three-state held-out evaluation (never used for selection)
        ev_ep, br_ep = evaluate_states(model, heldout_loader_ep, ['on', 'norm_only', 'off'], collect_branch=True)
        for st in ('on', 'norm_only', 'off'):
            m_ = ev_ep[st]
            EPOCH_ROWS.append({'arm': arm, 'seed': seed, 'epoch': epoch, 'state': st, **{k: v for k, v in m_.items() if k not in ('gold', 'pred', 'prob_yes')},
                               'adapter_alpha': float(model.adapter.alpha.detach().cpu()), 'mean_relative_branch_norm_cls': br_ep})
            for i, (g, p_, pr) in enumerate(zip(m_['gold'], m_['pred'], m_['prob_yes'])):
                EPOCH_PRED_ROWS.append({'arm': arm, 'seed': seed, 'epoch': epoch, 'state': st, 'example_index': i, 'passage_cluster': int(CLUSTER[i]), 'gold': int(g), 'prediction': int(p_), 'prob_yes': float(pr)})
        print(f"  held-out epoch {epoch}: on {100*ev_ep['on']['balanced_accuracy']:.2f} | norm_only {100*ev_ep['norm_only']['balanced_accuracy']:.2f} | off {100*ev_ep['off']['balanced_accuracy']:.2f}")
        model.train()
        key = (val['balanced_accuracy'], val['macro_f1'], -val['loss'])
        if best['key'] is None or key > best['key']: best = {'key': key, 'state': cpu_state_dict(model), 'epoch': epoch, 'metrics': {k: v for k, v in val.items() if k not in ('gold', 'pred', 'prob_yes')}}
    model.load_state_dict(best['state'], strict=True); model.to(DEVICE).eval(); del best['state']; gc.collect()
    return {'model': model, 'best_epoch': best['epoch'], 'best_val': best['metrics'], 'history': history, 'training_seconds': time.time() - start, 'peak_gpu_gb': torch.cuda.max_memory_allocated() / 1024 ** 3}

In [ ]:
# 7) Six restart-safe runs with per-run Drive commits, full-state checkpoints, three-state held-out evaluation and the STOP gate.
def read_partial(name):
    p = PROGRESS_DIR / name; return pd.read_csv(p).to_dict('records') if p.is_file() else []
runtime_rows = read_partial('runtime_partial.csv'); done = {(r['arm'], int(r['seed'])) for r in runtime_rows}
def committed(name): return [r for r in read_partial(name) if (r['arm'], int(r['seed'])) in done]
history_rows, run_rows, pred_rows, d4_rows, full_state_rows = committed('history_partial.csv'), committed('run_partial.csv'), committed('predictions_partial.csv'), committed('d4_partial.csv'), committed('full_state_partial.csv')
EPOCH_ROWS, EPOCH_PRED_ROWS = committed('epoch_partial.csv'), committed('epoch_predictions_partial.csv')
if done: print('Completed runs restored from Drive:', sorted(done))

def commit_all():
    for name, rows in (('history_partial.csv', history_rows), ('run_partial.csv', run_rows), ('predictions_partial.csv', pred_rows), ('d4_partial.csv', d4_rows), ('full_state_partial.csv', full_state_rows), ('epoch_partial.csv', EPOCH_ROWS), ('epoch_predictions_partial.csv', EPOCH_PRED_ROWS), ('runtime_partial.csv', runtime_rows)):
        pd.DataFrame(rows).to_csv(PROGRESS_DIR / name, index=False)

def stop_report(arm, seed, devs):
    rep = {'stop_gate': 'A.7.4', 'arm': arm, 'seed': seed, 'deviations_points': devs, 'threshold_points': STOP_GATE_POINTS,
           'instruction': 'Investigate (environment, library versions, data order, class weights, selection rule); write the finding to MANIFEST_LOCK.md annotations; then re-run — completed runs resume from Drive.'}
    (RESULT_DIR / 'STOP_GATE_REPORT.json').write_text(json.dumps(rep, indent=2)); shutil.copy2(RESULT_DIR / 'STOP_GATE_REPORT.json', PROGRESS_DIR / 'STOP_GATE_REPORT.json')
    return rep

for arm in ARMS:
    for seed in SEEDS:
        if (arm, seed) in done: print(f'{arm} seed {seed}: already complete, skipping.'); continue
        print(f'\n===== {arm.upper()} FULL FINE-TUNING, SEED {seed} =====')
        trained = train_run(arm, seed); model = trained['model']
        _, heldout_loader = make_loader(encoded['evaluation'], seed, training=False)
        states = ['on', 'norm_only', 'off'] if arm == 'lraa' else ['on']
        ev, branch_ratio = evaluate_states(model, heldout_loader, states, collect_branch=(arm == 'lraa'))
        for st in states:
            m = ev[st]
            run_rows.append({'arm': arm, 'seed': seed, 'state': st, 'best_epoch': trained['best_epoch'], **{k: v for k, v in m.items() if k not in ('gold', 'pred', 'prob_yes')},
                             'adapter_alpha': float(model.adapter.alpha.detach().cpu()) if model.adapter is not None else None})
            for i, (g, p, pr) in enumerate(zip(m['gold'], m['pred'], m['prob_yes'])):
                pred_rows.append({'arm': arm, 'seed': seed, 'state': st, 'example_index': i, 'passage_cluster': int(CLUSTER[i]), 'gold': int(g), 'prediction': int(p), 'prob_yes': float(pr)})
        if arm == 'lraa':
            d4_rows.append({'arm': arm, 'seed': seed, 'gate_value_sigmoid_alpha': float(model.adapter.alpha.detach().cpu()), 'mean_relative_branch_norm_cls': branch_ratio})
        # full model state to Drive with hash (encoder + adapter + scorer)
        local_state = WORK_DIR / f'fullft_{arm}_seed{seed}_full_state.pt'; torch.save({'arm': arm, 'seed': seed, 'best_epoch': trained['best_epoch'], 'model_state_dict': cpu_state_dict(model)}, local_state)
        digest = sha256_file(local_state); drive_state = FULL_STATE_DIR / local_state.name; shutil.copy2(local_state, drive_state)
        if sha256_file(drive_state) != digest: raise RuntimeError('Drive copy hash mismatch')
        full_state_rows.append({'arm': arm, 'seed': seed, 'file': str(drive_state), 'bytes': drive_state.stat().st_size, 'sha256': digest}); local_state.unlink()
        history_rows.extend(trained['history'])
        runtime_rows.append({'arm': arm, 'seed': seed, 'training_minutes': trained['training_seconds'] / 60, 'peak_allocated_gpu_memory_gb': trained['peak_gpu_gb'], 'gpu': GPU_NAME})
        commit_all(); done.add((arm, seed)); print(f'{arm} seed {seed} committed to Drive (best epoch {trained["best_epoch"]}).')
        # FT-TRAJ-1: no archived counterpart for these seeds; no STOP comparison (v1.3 A.12)
        del trained, model, ev; gc.collect(); torch.cuda.empty_cache()

RUN = pd.DataFrame(run_rows); PRED = pd.DataFrame(pred_rows); HIST = pd.DataFrame(history_rows); RT = pd.DataFrame(runtime_rows); D4 = pd.DataFrame(d4_rows); FS = pd.DataFrame(full_state_rows)
for df, name in ((RUN, 'fullft_run_results.csv'), (PRED, 'fullft_predictions.csv'), (HIST, 'training_history.csv'), (RT, 'runtime_by_run.csv'), (D4, 'd4_fullft_gate_and_branch_norm.csv'), (FS, 'full_state_checkpoint_manifest.csv')):
    df.to_csv(RESULT_DIR / name, index=False)
EPOCH = pd.DataFrame(EPOCH_ROWS); EPOCH_PRED = pd.DataFrame(EPOCH_PRED_ROWS); EPOCH.to_csv(RESULT_DIR / 'per_epoch_three_state_results.csv', index=False); EPOCH_PRED.to_csv(RESULT_DIR / 'per_epoch_three_state_predictions.csv', index=False)
assert len(RUN) == 3 * len(SEEDS), RUN
display(RUN.pivot_table(index=['arm', 'seed'], columns='state', values='balanced_accuracy').mul(100).round(2))

In [ ]:
# 8) FT-TRAJ-1 analysis: per-epoch dependence curves; A_FT of every full-fine-tuning trajectory; extended replication block (t(4) over five rerun seeds).
T = {2: 4.302652729911275, 4: 2.7764451051977987}
def ci(v):
    v = np.asarray(v, float); m = float(v.mean()); sd = float(v.std(ddof=1)); df = len(v) - 1; h = T.get(df, 4.302652729911275) * sd / math.sqrt(len(v)); return m, sd, m - h, m + h
def BAe(seed, epoch, st): return 100 * float(EPOCH[(EPOCH.seed == seed) & (EPOCH.epoch == epoch) & (EPOCH.state == st)].iloc[0]['balanced_accuracy'])
curve = []
for s_ in SEEDS:
    sel = int(RUN[(RUN.seed == s_)].iloc[0]['best_epoch'])
    for e in range(1, EPOCHS + 1):
        on, no, off = BAe(s_, e, 'on'), BAe(s_, e, 'norm_only'), BAe(s_, e, 'off')
        br = float(EPOCH[(EPOCH.seed == s_) & (EPOCH.epoch == e) & (EPOCH.state == 'on')].iloc[0]['mean_relative_branch_norm_cls'])
        curve.append({'seed': s_, 'epoch': e, 'selected': e == sel, 'on': on, 'norm_only': no, 'off': off, 'A_FT': on - no, 'B_FT': no - off, 'on_minus_off': on - off, 'branch_norm_cls': br,
                      'val_BA': 100 * float(HIST[(HIST.seed == s_) & (HIST.epoch == e)].iloc[0]['val_balanced_accuracy'])})
CURVE = pd.DataFrame(curve); CURVE.to_csv(RESULT_DIR / 'dependence_by_epoch.csv', index=False)

# all full-FT trajectories: archived control (on−off only; frozen-side B was 0 to within 0.2 pt), NB-2 rerun (three states), FT-TRAJ-1 (three states, selected epoch)
traj = []
for s_ in (13, 42, 71):
    traj.append({'source': 'archived control', 'seed': s_, 'selected_epoch': int(ARCH_V12[(ARCH_V12.seed == s_)].iloc[0]['best_epoch']), 'enabled_BA': ARCHIVED_FT_BA[(s_, 'on')], 'A_FT': None, 'on_minus_off': ARCHIVED_FT_BA[(s_, 'on')] - ARCHIVED_FT_BA[(s_, 'off')]})
nb2_progress = Path('/content/drive/MyDrive/FMI/NB2/progress/run_partial.csv')
if nb2_progress.is_file():
    R2 = pd.read_csv(nb2_progress); R2 = R2[R2.arm == 'lraa']
    for s_ in sorted(R2.seed.unique()):
        g = lambda st: 100 * float(R2[(R2.seed == s_) & (R2.state == st)].iloc[0]['balanced_accuracy'])
        traj.append({'source': 'NB-2 rerun', 'seed': int(s_), 'selected_epoch': int(R2[R2.seed == s_].iloc[0]['best_epoch']), 'enabled_BA': g('on'), 'A_FT': g('on') - g('norm_only'), 'on_minus_off': g('on') - g('off')})
for s_ in SEEDS:
    sel = int(RUN[(RUN.seed == s_)].iloc[0]['best_epoch']); r = CURVE[(CURVE.seed == s_) & (CURVE.epoch == sel)].iloc[0]
    traj.append({'source': 'FT-TRAJ-1', 'seed': s_, 'selected_epoch': sel, 'enabled_BA': r['on'], 'A_FT': r['A_FT'], 'on_minus_off': r['on_minus_off']})
TRAJ = pd.DataFrame(traj); TRAJ.to_csv(RESULT_DIR / 'all_fullft_trajectories.csv', index=False)

# extended replication block: five rerun seeds (NB-2 13/42/71 + FT-TRAJ-1), t(4)
ext = TRAJ[TRAJ.source.isin(['NB-2 rerun', 'FT-TRAJ-1'])]
summary = {}
if len(ext) >= 3:
    for q in ('A_FT', 'on_minus_off', 'enabled_BA'):
        m, sd, lo, hi = ci(ext[q].astype(float)); summary[q] = {'n_seeds': int(len(ext)), 'mean': m, 'sd': sd, 'ci95_low': lo, 'ci95_high': hi, 'values': ext[q].astype(float).round(3).tolist()}
# optimization-validity gate for the new seeds
best_val = HIST.groupby('seed')['val_balanced_accuracy'].max(); enabled = pd.Series({s_: float(RUN[(RUN.seed == s_) & (RUN.state == 'on')].iloc[0]['balanced_accuracy']) for s_ in SEEDS})
gate = {'all_seed_best_internal_val_BA_ge_0.60': bool((best_val >= 0.60).all()), 'all_seed_enabled_heldout_BA_above_chance': bool((enabled > 0.5).all()), 'mean_enabled_heldout_BA_exceeds_frozen_head_only': bool(enabled.mean() > FROZEN_HEAD_ONLY_MEAN_BA)}
gate['status'] = 'VALID' if all(gate.values()) else 'INVALID_OPTIMIZATION'
report = {'optimization_gate': gate, 'extended_replication_t4': summary, 'trajectories': TRAJ.to_dict('records'),
          'note': 'Descriptive; no threshold defines dependence. The distribution of A_FT across trajectories is reported in full, with enabled BA alongside.'}
(RESULT_DIR / 'ft_traj_analysis_summary.json').write_text(json.dumps(report, indent=2, default=float))
print('Per-epoch dependence (FT-TRAJ-1 seeds):'); display(CURVE.round(2))
print('\nAll full-fine-tuning trajectories:'); display(TRAJ.round(2))
print('\nExtended replication (rerun seeds, t(4)):', json.dumps(summary, indent=1, default=float)); print('Optimization gate:', gate)

In [ ]:
# 9) Environment, manifest, ZIP, Drive copy, download.
env = {'notebook': 'NB-2b FT-TRAJ-1 per-epoch', 'manifest': MANIFEST_ID, 'manifest_lock_commit': MANIFEST_LOCK_COMMIT, 'repository_commit': REPO_COMMIT,
       'gpu': GPU_NAME, 'amp_dtype': str(AMP_DTYPE), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'transformers': __import__('transformers').__version__,
       'datasets': __import__('datasets').__version__, 'gradient_checkpointing': True, 'full_states_on_drive': str(FULL_STATE_DIR),
       'bootstrap_seed': BOOTSTRAP_SEED, 'bootstrap_replicates': BOOTSTRAP_REPLICATES, 'nb1_source': (nb1_zips[-1].name if nb1_zips else 'embedded values'),
       'run_started_utc': RUN_STARTED_UTC, 'run_finished_utc': datetime.now(timezone.utc).isoformat(), 'session_active_seconds': time.perf_counter() - RUN_STARTED_PERF}
(RESULT_DIR / 'run_environment.json').write_text(json.dumps(env, indent=2))
rows = [{'file': p.name, 'bytes': p.stat().st_size, 'sha256': sha256_file(p)} for p in sorted(RESULT_DIR.iterdir()) if p.is_file() and p.name != 'FILE_MANIFEST_SHA256.csv']
pd.DataFrame(rows).to_csv(RESULT_DIR / 'FILE_MANIFEST_SHA256.csv', index=False)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); zip_path = Path(shutil.make_archive(f'/content/fmi_nb2b_ft_traj1_results_{stamp}', 'zip', RESULT_DIR))
with zipfile.ZipFile(zip_path) as zf:
    if zf.testzip(): raise RuntimeError('output ZIP corrupt')
shutil.copy2(zip_path, DRIVE_ROOT / zip_path.name); print('Result ZIP:', zip_path, '| sha256', sha256_file(zip_path), '| Drive copy:', DRIVE_ROOT / zip_path.name)
from google.colab import files
files.download(str(zip_path)); print('Done. Full-state checkpoints remain on Drive under', FULL_STATE_DIR)